# Corner-WFS Donut Selection Stages: Selected → Fit → Used (v1)

**Author:** Aaron Roodman
**Date Created:** 2026-06-13
**Last Modified:** 2026-06-13
**Status:** In Progress
**Keywords:** WFS, corner wavefront sensor, donut selection, ts_wep, zernikes, AOS

## Description

For each corner-WFS half-chip (intra / extra), show **where the donuts are** at each
stage of the ts_wep pipeline, and build a per-visit aggregate table of fit donuts with
a `used` flag.

ts_wep corner-WFS data products (per visit):
- `donutTable` (per detector) — **round-1 selected** donuts from direct detection
  (extra on SW0, intra on SW1).
- `aggregateDonutTable` (per visit) — the donuts carried forward into fitting
  (extra side capped per detector); all fit donuts appear here.
- `aggregateAOSVisitTableRaw` (per visit) — one row **per donut pair** that was
  **fit**, concatenated over the SW0/extra detectors, with a **`used`** flag
  (`used==True` ⇒ pair used in the averaged Zernike estimate). Each row links
  `donut_id_intra` and `donut_id_extra`.

Stage definitions used here (join key = `donut_id`):
- **selected** — in `donutTable` (round-1 direct detection).
- **fit** — donut is a member of a pair in `aggregateAOSVisitTableRaw`.
- **used** — donut is a member of a pair with `used == True`.

**Output:** A per-visit PDF of per-CCD donut-location plots and an extended
`aggregateDonutTable` (with `fit`/`used` columns) written to `wfs/output/`.

**Based on:** AOS `cwfs` reductions in
`LSSTCam/runs/aos/cwfs/danish_1_0/wep_17_3_0/dv_4_2_0/bin_x2` (repo `/repo/main`).

## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-06-13 | Aaron Roodman | Initial version: per-CCD donut-location plots (selected/fit/used) + extended aggregate table |

## Table of Contents

1. [Parameters](#params)
2. [Setup & Imports](#setup)
3. [Helper Functions](#functions)
4. [Data Access](#data)
5. [Select Visits](#select)
6. [Per-CCD Donut-Location Plots](#plots)
7. [Aggregate Table (fit donuts + used flag)](#table)

<a id='params'></a>
## Parameters

In [ ]:
# ============================================================
# Parameters — All configurable values collected here
# ============================================================

butler_repo = "/repo/main"
collection = "LSSTCam/runs/aos/cwfs/danish_1_0/wep_17_3_0/dv_4_2_0/bin_x2"
instrument = "LSSTCam"

day_obs = 20260327               # night to inspect
max_visits = 3                   # how many visits to render (None = all on the night)
start_index = 0

# Corner wavefront-sensor half-chips. SW0 = extra-focal, SW1 = intra-focal;
# the paired fit products (zernikes / aggregateAOSVisitTableRaw) live on SW0.
WFS_DETECTORS = {
    191: "R00_SW0", 192: "R00_SW1",
    195: "R04_SW0", 196: "R04_SW1",
    199: "R40_SW0", 200: "R40_SW1",
    203: "R44_SW0", 204: "R44_SW1",
}
DEFOCAL = {"SW0": "extra", "SW1": "intra"}

# Marker styling per stage
style_selected = dict(marker="o", mfc="none", mec="0.6", ms=7, mew=1.0, ls="none")
style_fit      = dict(marker="o", mfc="none", mec="tab:orange", ms=9, mew=1.6, ls="none")
style_used     = dict(marker="o", mfc="tab:green", mec="k", ms=7, mew=0.6, ls="none",
                      alpha=0.85)

# Output
output_dir = "output"
output_pdf = None                # None -> output/wfs_donut_stages_{day_obs}.pdf
save_table = True                # write extended aggregate table (ECSV)
panel_width = 5.0
title_fontsize = 9
dpi = 150


<a id='setup'></a>
## Setup & Imports

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from astropy.table import vstack

import lsst.daf.butler as dafButler

sys.path.insert(0, str(Path.cwd().parent))
from common.utils import setup_plotting

setup_plotting()

os.makedirs(output_dir, exist_ok=True)


<a id='functions'></a>
## Helper Functions

In [ ]:
def list_visits(butler, collection, instrument, day_obs):
    """Visits on a night that have an aggregateAOSVisitTableRaw (i.e. were fit)."""
    refs = butler.query_datasets(
        "aggregateAOSVisitTableRaw", collections=collection,
        where=f"instrument=\'{instrument}\' and exposure.day_obs={day_obs}",
        explain=False, limit=None,
    )
    return sorted({r.dataId["visit"] for r in refs})


def stage_sets(butler, collection, instrument, visit):
    """Return (fit_ids, used_ids): sets of donut_id that were fit / used.

    Built from aggregateAOSVisitTableRaw (one row per fit pair, with `used`).
    """
    ar = butler.get("aggregateAOSVisitTableRaw", collections=collection,
                    dataId={"instrument": instrument, "visit": visit})
    intra = np.asarray(ar["donut_id_intra"])
    extra = np.asarray(ar["donut_id_extra"])
    used = np.asarray(ar["used"]).astype(bool)
    fit_ids = set(intra) | set(extra)
    used_ids = set(intra[used]) | set(extra[used])
    return fit_ids, used_ids


def load_donut_table(butler, collection, instrument, visit, detector):
    """Round-1 selected donuts (direct detect) for one detector, or None."""
    try:
        return butler.get("donutTable", collections=collection,
                          dataId={"instrument": instrument, "visit": visit,
                                  "detector": detector})
    except Exception:
        return None


def classify(donut_ids, fit_ids, used_ids):
    """Per-donut stage label array: 'used' | 'fit' | 'selected'."""
    out = np.full(len(donut_ids), "selected", dtype=object)
    for i, d in enumerate(donut_ids):
        if d in used_ids:
            out[i] = "used"
        elif d in fit_ids:
            out[i] = "fit"
    return out


def extended_aggregate(butler, collection, instrument, visit, fit_ids, used_ids):
    """aggregateDonutTable for a visit, with `visit`, `fit`, `used` columns added."""
    agg = butler.get("aggregateDonutTable", collections=collection,
                     dataId={"instrument": instrument, "visit": visit})
    ids = np.asarray(agg["donut_id"])
    agg["visit"] = visit
    agg["fit"] = np.array([d in fit_ids for d in ids])
    agg["used"] = np.array([d in used_ids for d in ids])
    return agg


<a id='data'></a>
## Data Access

In [ ]:
butler = dafButler.Butler(butler_repo, collections=collection)
print("Repo:", butler_repo)
print("Collection:", collection)


<a id='select'></a>
## Select Visits

In [ ]:
all_visits = list_visits(butler, collection, instrument, day_obs)
print(f"day_obs {day_obs}: {len(all_visits)} fit visits")
visits = all_visits[start_index:]
if max_visits is not None:
    visits = visits[:max_visits]
print("Rendering visits:", visits)


<a id='plots'></a>
## Per-CCD Donut-Location Plots

In [ ]:
rafts = sorted({n.split("_")[0] for n in WFS_DETECTORS.values()})
name_to_det = {v: k for k, v in WFS_DETECTORS.items()}

legend_handles = [
    Line2D([0], [0], label="selected (round 1)", **style_selected),
    Line2D([0], [0], label="fit (in a pair)", **style_fit),
    Line2D([0], [0], label="used (used pair)", **style_used),
]


def plot_visit_locations(visit, fit_ids, used_ids):
    """4x2 grid of per-half-chip donut centroids, coloured by stage."""
    panel_h = panel_width * 2000.0 / 4072.0
    fig, axes = plt.subplots(len(rafts), 2,
                             figsize=(2 * panel_width, len(rafts) * panel_h),
                             constrained_layout=True)
    axes = np.atleast_2d(axes)
    fig.set_constrained_layout_pads(w_pad=0.02, h_pad=0.02, wspace=0.03, hspace=0.06)

    for i, raft in enumerate(rafts):
        for j, sw in enumerate(("SW0", "SW1")):
            ax = axes[i, j]; name = f"{raft}_{sw}"; det = name_to_det[name]
            dt = load_donut_table(butler, collection, instrument, visit, det)
            n = dict(selected=0, fit=0, used=0)
            if dt is not None and len(dt):
                ids = np.asarray(dt["donut_id"])
                x = np.asarray(dt["centroid_x"]); y = np.asarray(dt["centroid_y"])
                lab = classify(ids, fit_ids, used_ids)
                for stage, st in (("selected", style_selected),
                                  ("fit", style_fit), ("used", style_used)):
                    m = lab == stage
                    n[stage] = int(m.sum())
                    if m.any():
                        ax.plot(x[m], y[m], **st)
            ax.set_xlim(0, 4072); ax.set_ylim(0, 2000); ax.set_aspect("equal")
            ax.set_title(f"{name} ({det}, {DEFOCAL[sw]})  "
                         f"sel={n['selected']+n['fit']+n['used']} "
                         f"fit={n['fit']+n['used']} used={n['used']}",
                         fontsize=title_fontsize)
            ax.set_xticks([]); ax.set_yticks([])

    fig.legend(handles=legend_handles, loc="upper right", fontsize=8, ncol=3)
    fig.suptitle(f"Visit {visit} — corner-WFS donut selection stages",
                 fontsize=title_fontsize + 3)
    return fig


from matplotlib.backends.backend_pdf import PdfPages
pdf_path = Path(output_pdf) if output_pdf else \
    Path(output_dir) / f"wfs_donut_stages_{day_obs}.pdf"

agg_tables = []
with PdfPages(pdf_path) as pdf:
    for visit in visits:
        fit_ids, used_ids = stage_sets(butler, collection, instrument, visit)
        fig = plot_visit_locations(visit, fit_ids, used_ids)
        pdf.savefig(fig, dpi=dpi)
        plt.show()
        agg_tables.append(extended_aggregate(butler, collection, instrument, visit,
                                             fit_ids, used_ids))
        print(f"visit {visit}: fit donuts={len(fit_ids)}, used donuts={len(used_ids)}")

print(f"\nWrote {len(visits)} page(s) to {pdf_path}")


<a id='table'></a>
## Aggregate Table (fit donuts + used flag)

In [ ]:
# Extended aggregateDonutTable for the rendered visits, with fit/used flags.
aggregate = vstack(agg_tables, metadata_conflicts="silent") if agg_tables else None

if aggregate is not None:
    nfit = int(np.sum(aggregate["fit"])); nused = int(np.sum(aggregate["used"]))
    print(f"aggregate rows: {len(aggregate)}  (fit={nfit}, used={nused})")
    # The "final list of donuts which were fit":
    fit_donuts = aggregate[aggregate["fit"]]
    cols = ["visit", "detector", "donut_id", "centroid_x", "centroid_y",
            "thx_OCS", "thy_OCS", "snr", "fit", "used"]
    cols = [c for c in cols if c in aggregate.colnames]
    print(fit_donuts[cols][:12])

    if save_table:
        out = Path(output_dir) / f"wfs_donut_aggregate_{day_obs}.ecsv"
        aggregate.write(out, overwrite=True)
        print("\nWrote table:", out)
